# Chess.com → Stockfish → errors_by_stage

In [72]:
!pip install python-chess requests tqdm
!apt-get install -y stockfish

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
stockfish is already the newest version (14.1-1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [73]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/InfoVis/tp-chess/chess_data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [74]:
import requests, chess, chess.pgn, chess.engine, pandas as pd, io, os
from collections import defaultdict
from tqdm import tqdm

USERNAME = "sugeema"
ENGINE_TIME = 0.03

In [79]:
import requests
import time

USERNAME = "sugeema"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ChessDataBot/1.0)"
}

def get_pgn_urls(username):
    url = f"https://api.chess.com/pub/player/{username}/games/archives"

    res = requests.get(url, headers=HEADERS)

    if res.status_code != 200:
        print("❌ Error al obtener archivos:", res.status_code)
        print(res.text[:300])
        return []

    try:
        data = res.json()
        return data["archives"]
    except Exception as e:
        print("❌ Error parseando JSON:")
        print(res.text[:300])
        return []


def download_pgns(username, max_archives=2):
    archives = get_pgn_urls(username)

    if not archives:
        print("⚠️ No se encontraron archivos")
        return ""

    print(f"📦 Total archivos disponibles: {len(archives)}")
    print(f"⬇️ Descargando últimos {max_archives}...")

    pgn_texts = []

    for url in archives[-max_archives:]:
        print(f"Descargando: {url}")

        res = requests.get(url + "/pgn", headers=HEADERS)

        if res.status_code == 200:
            pgn_texts.append(res.text)
        else:
            print("❌ Error PGN:", res.status_code)
            print(res.text[:200])

        time.sleep(1)  # evita rate limit

    full_pgn = "\n\n".join(pgn_texts)

    print("✅ Descarga completa")
    print("Preview:")
    print(full_pgn[:500])

    return full_pgn


# Ejecutar
pgn_data = download_pgns(USERNAME, max_archives=66)

📦 Total archivos disponibles: 66
⬇️ Descargando últimos 66...
Descargando: https://api.chess.com/pub/player/sugeema/games/2020/11
Descargando: https://api.chess.com/pub/player/sugeema/games/2020/12
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/01
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/02
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/03
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/04
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/05
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/06
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/07
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/08
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/09
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/10
Descargando: https://api.chess.com/pub/player/sugeema/games/2021/11
Descargando: https://api.chess.com/pub/player/sugeema/

In [80]:
pgn_io = io.StringIO(pgn_data)
games = []
while True:
    g = chess.pgn.read_game(pgn_io)
    if g is None:
        break
    games.append(g)
print(len(games))

453


In [81]:
engine = chess.engine.SimpleEngine.popen_uci("/usr/games/stockfish")

In [82]:
def evaluate(board):
    info = engine.analyse(board, chess.engine.Limit(time=ENGINE_TIME))
    score = info["score"].white().score(mate_score=10000)
    return score if score else 0

"""
Esta función define los blunders, mistakes e inaccuracies
"""
def classify(delta):
    if delta <= -300: return "blunders"
    elif delta <= -100: return "mistakes"
    elif delta <= -50: return "inaccuracies"
    return None


"""
Esta función define si es juego final, en funcion de cantidad de piezas o si falta la dama
"""
def is_endgame(board):
    pieces = len(board.piece_map())

    queens = sum(1 for piece in board.piece_map().values() if piece.piece_type == chess.QUEEN)

    return pieces <= 16 or queens == 0


"""
Esta función define las etapas de apertura, juego medio y final
"""
def get_stage(move_number, board):
    if move_number <= 15:
        return "opening"
    elif is_endgame(board):
        return "endgame"
    else:
        return "middlegame"

In [83]:
stats = defaultdict(lambda: {"blunders":0, "mistakes":0, "inaccuracies":0, "moves":0, "games":0})

for game in tqdm(games):
    board = game.board()
    is_white = game.headers["White"].lower() == USERNAME.lower()

    stages_seen = set()

    for move_number, move in enumerate(game.mainline_moves(), start=1):
        is_my_turn = (board.turn and is_white) or ((not board.turn) and (not is_white))
        before = evaluate(board)
        board.push(move)
        after = evaluate(board)
        if is_my_turn:
            delta = after - before
            if not is_white:
                delta = -delta
            stage = get_stage(move_number, board)
            et = classify(delta)
            if et:
                stats[stage][et] += 1
            stats[stage]["moves"] += 1
            stages_seen.add(stage)

    for stage in stages_seen:
        stats[stage]["games"] += 1

100%|██████████| 453/453 [28:51<00:00,  3.82s/it]


In [85]:
rows = []
for stage, data in stats.items():
    for t in ["blunders", "mistakes", "inaccuracies"]:
        rows.append({
            "stage": stage,
            "type": t,
            "count": data[t],
            "games": data["games"],
            "avg": data[t] / data["games"] if data["games"] else 0
        })
df = pd.DataFrame(rows)
df

,stage,type,count,games,avg
0,opening,blunders,166,453,0.366446
1,opening,mistakes,522,453,1.152318
2,opening,inaccuracies,431,453,0.951435
3,middlegame,blunders,1113,422,2.637441
4,middlegame,mistakes,1453,422,3.443128
5,middlegame,inaccuracies,891,422,2.111374
6,endgame,blunders,342,211,1.620853
7,endgame,mistakes,552,211,2.616114
8,endgame,inaccuracies,443,211,2.099526


In [86]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Agregar games y counts al wide
wide = df.pivot(index="stage", columns="type", values="avg").reset_index()
wide.columns = ["stage", "blunders_avg", "inaccuracies_avg", "mistakes_avg"]

counts = df.pivot(index="stage", columns="type", values="count").reset_index()
counts.columns = ["stage", "blunders_count", "inaccuracies_count", "mistakes_count"]

games = df[["stage", "games"]].drop_duplicates()

wide = wide.merge(counts, on="stage").merge(games, on="stage")

# Long ya tiene todo
long_path = f"{OUTPUT_DIR}/errors_by_stage_long.csv"
df.to_csv(long_path, index=False)

wide_path = f"{OUTPUT_DIR}/errors_by_stage_wide.csv"
wide.to_csv(wide_path, index=False)

print(long_path, wide_path)

/content/drive/MyDrive/InfoVis/tp-chess/chess_data/errors_by_stage_long.csv /content/drive/MyDrive/InfoVis/tp-chess/chess_data/errors_by_stage_wide.csv


In [87]:
engine.quit()

In [88]:
for stage, data in stats.items():
    print(stage)
    print("moves:", data["moves"])
    print("blunders:", data["blunders"])
    print("mistakes:", data["mistakes"])
    print("inaccuracies:", data["inaccuracies"])
    print()

opening
moves: 3318
blunders: 166
mistakes: 522
inaccuracies: 431

middlegame
moves: 6558
blunders: 1113
mistakes: 1453
inaccuracies: 891

endgame
moves: 3283
blunders: 342
mistakes: 552
inaccuracies: 443



In [89]:
for stage, data in stats.items():
    print(stage, "→ games:", data["games"], "| moves:", data["moves"])

opening → games: 453 | moves: 3318
middlegame → games: 422 | moves: 6558
endgame → games: 211 | moves: 3283


In [93]:
wide2 = df.pivot(index="type", columns="stage", values="avg").reset_index()
wide2["type"] = pd.Categorical(wide2["type"], categories=["blunders", "mistakes", "inaccuracies"], ordered=True)
wide2 = wide2.sort_values("type").reset_index(drop=True)
wide2 = wide2[["type", "opening", "middlegame", "endgame"]]
wide2_path = f"{OUTPUT_DIR}/errors_by_stage_grouped.csv"
wide2.to_csv(wide2_path, index=False)
print(wide2_path)

/content/drive/MyDrive/InfoVis/tp-chess/chess_data/errors_by_stage_grouped.csv
